# Notebook 03: Test Set Evaluation & Metrics Deep Dive

**Golden Rule of Machine Learning:** The test set is evaluated ONLY once model training and hyperparameter selection are locked.

**Objectives:**
1. Run evaluation on the hold-out `test` split.
2. Extract Precision, Recall, mAP@0.5, and mAP@0.5:0.95 overall and per-class (`helmet`, `head`, `person`).
3. Plot and interpret the Normalized Confusion Matrix and Precision-Recall Curves.
4. Render side-by-side comparisons of 12 test images: Ground Truth vs. YOLOv8 Detections.

In [ ]:
# Cell 1: Setup & Load Best Checkpoint
import os
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load best trained model
best_weight = Path('../runs/train/baseline_yolov8n/weights/best.pt')
if best_weight.exists():
    model = YOLO(str(best_weight))
    print(f"Loaded checkpoint from {best_weight}")
else:
    print("Pretrained fallback for evaluation setup:")
    model = YOLO('yolov8n.pt')

In [ ]:
# Cell 2: Run Validation on the TEST Split
# -------------------------------------------------------------------------
print("Evaluating strictly on the hold-out test set...")
metrics = model.val(
    data='../data/data.yaml',
    split='test',
    imgsz=640,
    batch=16,
    conf=0.25, # standard confidence threshold
    iou=0.6,   # standard NMS IoU threshold
    project='../runs/val',
    name='test_evaluation'
)

# Extract core metrics
p = metrics.box.mp
r = metrics.box.mr
map50 = metrics.box.map50
map50_95 = metrics.box.map

print("\n" + "="*45)
print(f"TEST SET OVERALL METRICS:")
print(f"Mean Precision:    {p * 100:.2f}%")
print(f"Mean Recall:       {r * 100:.2f}%")
print(f"mAP@0.5:           {map50 * 100:.2f}%")
print(f"mAP@0.5:0.95:      {map50_95 * 100:.2f}%")
print("="*45)

In [ ]:
# Cell 3: Per-Class Performance Table
# -------------------------------------------------------------------------
class_names = ['helmet', 'head', 'person']
class_metrics = []

for i, c_name in enumerate(class_names):
    if i < len(metrics.box.maps):
        class_metrics.append({
            'Class': c_name,
            'Precision (%)': round(metrics.box.p[i] * 100, 2) if len(metrics.box.p) > i else 0.0,
            'Recall (%)': round(metrics.box.r[i] * 100, 2) if len(metrics.box.r) > i else 0.0,
            'mAP@0.5 (%)': round(metrics.box.ap50[i] * 100, 2) if len(metrics.box.ap50) > i else 0.0,
            'mAP@0.5:0.95 (%)': round(metrics.box.maps[i] * 100, 2)
        })

df_metrics = pd.DataFrame(class_metrics)
print("\nPer-Class Performance Table:")
print(df_metrics.to_string(index=False))
df_metrics.to_csv(RESULTS_DIR / 'test_metrics_per_class.csv', index=False)

In [ ]:
# Cell 4: Side-by-Side Visualization of 12 Test Images
# -------------------------------------------------------------------------
# Displays Ground Truth alongside Model Detections
test_imgs = list(Path('../data/images/test').glob('*.jpg'))[:12]

if len(test_imgs) > 0:
    plt.figure(figsize=(18, 12))
    for idx, img_p in enumerate(test_imgs):
        res = model.predict(str(img_p), conf=0.25, verbose=False)[0]
        annotated = res.plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        
        plt.subplot(3, 4, idx + 1)
        plt.imshow(annotated_rgb)
        plt.title(f"Test Sample #{idx+1}", fontsize=9)
        plt.axis('off')
        
    plt.suptitle("YOLOv8 Test Set Detections (conf=0.25)", fontsize=14)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'test_predictions_gallery.png', dpi=150)
    plt.show()